# E0A controlled post-commit recovery

In [ ]:

from __future__ import annotations
import os, json, math, hashlib, random
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_MODE="debug"
env_root=os.environ.get("CERA_ROOT")
candidates=[Path(env_root)] if env_root else []
candidates += [Path("/kaggle/working/cera"),Path("/mnt/data/cera"),Path.cwd()]
ROOT=next((p for p in candidates if p.exists() and (p.name=="cera" or (p/"configs").exists())),Path.cwd())
BENCHMARK_ROOT=ROOT/("benchmark_debug" if RUN_MODE=="debug" else "benchmark")
RESULTS=ROOT/"results"/"E0A"; FIGURES=ROOT/"figures"/"E0A"
RESULTS.mkdir(parents=True,exist_ok=True); FIGURES.mkdir(parents=True,exist_ok=True)
if RUN_MODE not in {"debug","full"}: raise ValueError("RUN_MODE must be 'debug' or 'full'.")
TASKS_MANIFEST=BENCHMARK_ROOT/"manifests"/"e0a_task_ids.json"
CASE_MANIFEST=BENCHMARK_ROOT/"manifests"/"b0_case_manifest.jsonl"
if not TASKS_MANIFEST.exists() or not CASE_MANIFEST.exists():
    raise FileNotFoundError("Run generate_cera_bench.ipynb successfully first.")
target_task_ids=json.loads(TASKS_MANIFEST.read_text())
case_rows=[json.loads(x) for x in CASE_MANIFEST.read_text().splitlines() if x.strip()]
case_map={r["scenario_id"]:r for r in case_rows}
models=["qwen3-8b","qwen3-14b","qwen3-32b","llama3.1-8b-instruct"]
seeds=[42,43] if RUN_MODE=="full" else [42]
expected=len(target_task_ids)*len(models)*len(seeds)
print(f"mode={RUN_MODE} tasks={len(target_task_ids)} models={len(models)} seeds={len(seeds)} expected={expected}")


In [ ]:

def load_case(task_id,model,seed):
    sid=f"{task_id}__{model}__{seed}"
    row=case_map.get(sid)
    if row is None: raise RuntimeError(f"Missing frozen case: {sid}")
    trace=json.loads(Path(row["trace_path"]).read_text())
    oracle=json.loads(Path(row["oracle_path"]).read_text())
    return trace,oracle

def wilson(successes,n,z=1.96):
    if n==0:return (float("nan"),float("nan"))
    p=successes/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return c-h,c+h

def summarize_case(trace,oracle):
    labels=list(oracle["compatibility"].values())
    adopt=sum(x=="ADOPT" for x in labels); block=sum(x=="BLOCK" for x in labels)
    unknown=sum(x=="UNKNOWN" for x in labels)
    mixed=int(adopt>0 and block>0)
    action_div=bool(trace["action_divergence"]); effect_div=bool(trace["effect_relevant_divergence"])
    if effect_div and not action_div: raise RuntimeError(f"Effect/action invariant failed: {trace['scenario_id']}")
    if trace["target_regime"] and not effect_div: raise RuntimeError(f"Target/effect invariant failed: {trace['scenario_id']}")
    return {
        "scenario_id":trace["scenario_id"],"task_id":trace["task_id"],"template_id":trace["template_id"],
        "family":trace["family"],"model":trace["model"],"seed":trace["seed"],
        "fault_step":trace["fault_step"],"horizon_K":trace["horizon_K"],
        "action_divergence":action_div,"effect_relevant_divergence":effect_div,
        "historical_effects":len(labels),"adopt_effects":adopt,"block_effects":block,"unknown_effects":unknown,
        "mixed_compatibility":bool(mixed),"target_regime":bool(mixed),
        "adoption_opportunity":adopt/(adopt+block) if adopt+block else np.nan,
        "counterbalanced_margin":trace["counterbalanced_margin"],"prob_continue":trace["prob_continue"],
        "prob_switch":trace["prob_switch"],"label_bias":trace["label_bias"],"recovery_branch":trace["recovery_branch"],
        "source":trace["source"],"fallback":trace["fallback"],
        "valid_work":sum(e["utility"] for e in trace["history"] if oracle["compatibility"][e["effect_id"]]=="ADOPT"),
        "unsafe_reuse":sum(oracle["compatibility"][e["effect_id"]]=="BLOCK" for e in trace["history"] if oracle["compatibility"][e["effect_id"]]=="BLOCK"),
    }

records=[]
for i,task_id in enumerate(target_task_ids):
    for model in models:
        for seed in seeds:
            trace,oracle=load_case(task_id,model,seed)
            records.append(summarize_case(trace,oracle))
            print(f"[{len(records)}/{expected}] {trace['scenario_id']}",flush=True)
df=pd.DataFrame(records)
if len(df)!=expected: raise RuntimeError(f"Expected {expected} cases, found {len(df)}")


In [ ]:

coverage=df.groupby(["task_id","model","seed"]).size()
if coverage.max()!=1 or coverage.min()!=1: raise RuntimeError("Duplicate or missing task-model-seed cases.")
if df.task_id.nunique()!=len(target_task_ids): raise RuntimeError("Task coverage mismatch.")
if RUN_MODE=="full" and set(df.family)!={"software","deployment","data","transactional"}: raise RuntimeError("Family coverage mismatch.")
if not np.isfinite(df.counterbalanced_margin.astype(float)).all(): raise RuntimeError("Non-finite semantic margins.")
if not ((df.prob_continue>0.001)&(df.prob_continue<0.999)).all(): raise RuntimeError("Degenerate semantic probabilities.")
if df.fallback.any(): raise RuntimeError("A frozen benchmark case contains a fallback.")
if (df.source=="test_backend").any() and os.environ.get("CERA_TEST_MODE","0")!="1": raise RuntimeError("Unexpected test backend in non-test execution.")
target_families=set(df.loc[df.target_regime,"family"])
if len(target_families)<3 and RUN_MODE=="full": raise RuntimeError("Fewer than three workload families exhibit mixed compatibility.")
qwen=df[df.model=="qwen3-8b"]
diagnostic={
    "cases":len(qwen),
    "fallbacks":int(qwen.fallback.sum()),
    "both_branches_observed":qwen.recovery_branch.nunique()==2,
    "margin_std":float(qwen.counterbalanced_margin.std(ddof=1)) if len(qwen)>1 else 0.0,
    "prob_std":float(qwen.prob_continue.std(ddof=1)) if len(qwen)>1 else 0.0,
    "mean_abs_margin":float(qwen.counterbalanced_margin.abs().mean()),
    "mean_abs_label_bias":float(qwen.label_bias.abs().mean()),
    "mean_abs_bias_ratio":float(qwen.counterbalanced_margin.abs().mean()/max(qwen.label_bias.abs().mean(),1e-12)),
}
gate={
    "case_count":len(df)==expected,
    "task_coverage":df.task_id.nunique()==len(target_task_ids),
    "model_coverage":set(df.model)==set(models),
    "seed_coverage":set(df.seed)==set(seeds),
    "four_families":(set(df.family)=={"software","deployment","data","transactional"}) if RUN_MODE=="full" else True,
    "finite_margins":bool(np.isfinite(df.counterbalanced_margin).all()),
    "nondegenerate_probabilities":bool(((df.prob_continue>0.001)&(df.prob_continue<0.999)).all()),
    "no_fallbacks":not bool(df.fallback.any()),
    "effect_implies_action":bool((~df.effect_relevant_divergence|df.action_divergence).all()),
    "target_subset_effect":bool((~df.target_regime|df.effect_relevant_divergence).all()),
    "qwen_both_branches":diagnostic["both_branches_observed"] if RUN_MODE=="full" and os.environ.get("CERA_TEST_MODE","0")!="1" else True,
}
if not all(gate.values()): raise RuntimeError("E0A gate failed: "+", ".join(k for k,v in gate.items() if not v))


In [ ]:

family=df.groupby("family").agg(cases=("scenario_id","count"),divergence=("action_divergence","mean"),mixed=("mixed_compatibility","mean"),adoption=("adoption_opportunity","mean")).reset_index()
model=df.groupby("model").agg(cases=("scenario_id","count"),divergence=("action_divergence","mean"),mixed=("mixed_compatibility","mean"),adoption=("adoption_opportunity","mean")).reset_index()
workload=df.groupby("template_id").agg(cases=("scenario_id","count"),mixed=("mixed_compatibility","mean"),divergence=("action_divergence","mean"),adoption=("adoption_opportunity","mean")).reset_index()
metrics={
    "cases":int(len(df)),
    "divergence_rate":float(df.action_divergence.mean()),
    "mixed_compatibility_rate":float(df.mixed_compatibility.mean()),
    "mean_adoption_opportunity":float(df.adoption_opportunity.dropna().mean()),
    "mean_valid_work_retained":float(df.valid_work.mean()),
    "mean_unsafe_reuse":float(df.block_effects.mean()),
    "target_regime_families":sorted(target_families),
    "wilson_mixed":wilson(int(df.mixed_compatibility.sum()),len(df)),
}
df.to_csv(RESULTS/"case_results.csv",index=False)
family.to_csv(RESULTS/"family_summary.csv",index=False)
model.to_csv(RESULTS/"model_summary.csv",index=False)
workload.to_csv(RESULTS/"workload_summary.csv",index=False)
pd.DataFrame([metrics]).to_csv(RESULTS/"metrics.csv",index=False)
(RESULTS/"qwen_diagnostic.json").write_text(json.dumps(diagnostic,indent=2),encoding="utf-8")
(RESULTS/"production_gate.json").write_text(json.dumps({"gate":gate,"strict_pass":all(gate.values()),"mode":RUN_MODE},indent=2),encoding="utf-8")
(RESULTS/"summary.json").write_text(json.dumps(metrics,indent=2),encoding="utf-8")


In [ ]:

fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar(["Action divergence","Mixed compatibility","Target regime"],[df.action_divergence.mean(),df.mixed_compatibility.mean(),df.target_regime.mean()])
ax.set_ylim(0,1); ax.set_ylabel("Rate"); fig.tight_layout(); fig.savefig(FIGURES/"e0a_core_rates.png",dpi=180,bbox_inches="tight"); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5))
for i,m in enumerate(models):
    s=df[df.model==m]
    ax.bar(i,s.mixed_compatibility.mean())
ax.set_xticks(range(len(models))); ax.set_xticklabels(models,rotation=20,ha="right"); ax.set_ylabel("Mixed-compatibility rate"); fig.tight_layout(); fig.savefig(FIGURES/"e0a_model_mixed_compatibility.png",dpi=180,bbox_inches="tight"); plt.show()
qwen=df[df.model=="qwen3-8b"]
fig,ax=plt.subplots(figsize=(8,4.5)); ax.hist(qwen.counterbalanced_margin,bins=15); ax.axvline(0,linestyle="--"); ax.set_xlabel("Counterbalanced semantic margin"); ax.set_ylabel("Cases"); fig.tight_layout(); fig.savefig(FIGURES/"e0a_qwen_margin.png",dpi=180,bbox_inches="tight"); plt.show()
fig,ax=plt.subplots(figsize=(8,4.5)); ax.hist(qwen.prob_continue,bins=15); ax.axvline(.5,linestyle="--"); ax.set_xlabel("P(continue)"); ax.set_ylabel("Cases"); fig.tight_layout(); fig.savefig(FIGURES/"e0a_qwen_probability.png",dpi=180,bbox_inches="tight"); plt.show()
print(df[["action_divergence","mixed_compatibility","adoption_opportunity","valid_work","block_effects"]].describe())
print(json.dumps(metrics,indent=2))
